In [ ]:
# Make this notebook work from fine-tuning/ or fine-tuning/clinical-rtor/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('clinical-rtor', 'pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 00 (Clinical) · Build the Return-to-OR abstraction dataset

**Use case 2 — surgical-quality document abstraction.** A registry abstractor reads a patient timeline plus operative notes and decides whether a surgery is an *unplanned Return to the Operating Room (RTOR)* — and must **cite the exact sentence** that justifies the call. This lab turns a one-page rules file (`data/rtor_rules.md`) plus a handful of labeled cases into a full supervised-fine-tuning set.

> All op-notes, timelines, and NPIs here are **fully synthetic and PHI-free.**

*Demo moment:* a markdown rules file + 16 labeled cases become a train / validation / eval split the next labs train and score against.

---
## Step 0 — Preflight (run me first)

A self-contained green-light check: local data files, SDKs, your endpoint, `az login`, and a tiny call to the base model. It **only hard-fails** on missing local files — the Azure checks are advisory so the offline data path still runs.

In [ ]:
import os, sys, json
from pathlib import Path

_OK, _WARN, _FAIL = '[ OK ]', '[WARN]', '[FAIL]'
_problems = []

def _say(tag, msg):
    print(f'{tag} {msg}')

print('=== Preflight: Return-to-OR labs ===\n')

# 1) Local data files (hard requirement) -----------------------------------
need = ['data/rtor_rules.md', 'data/rtor_cases.jsonl', 'data/rtor_tools_schema.json']
missing = [f for f in need if not Path(f).exists()]
if missing:
    for f in missing:
        _say(_FAIL, f'missing {f}')
    _problems.append('data files')
else:
    n = sum(1 for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip())
    _say(_OK, f'data files present ({n} labeled cases)')

# 2) SDK imports (hard requirement) ----------------------------------------
try:
    import openai
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential
    _say(_OK, f'SDKs importable (openai {openai.__version__})')
except Exception as e:
    _say(_FAIL, f'SDK import failed: {e}  ->  pip install -r fine-tuning/requirements.txt')
    _problems.append('sdk')

# 3) Endpoint env var (needed for any Azure call) --------------------------
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
endpoint = os.environ.get('AZURE_OPENAI_ENDPOINT')
if endpoint:
    _say(_OK, f'AZURE_OPENAI_ENDPOINT = {endpoint}')
else:
    _say(_WARN, 'AZURE_OPENAI_ENDPOINT not set  ->  Labs 01/03/07 need it (set it or run setup-foundry.ps1)')

# 4) AAD token via az login (needed for any Azure call) --------------------
_token_ok = False
if 'DefaultAzureCredential' in dir() and endpoint:
    try:
        _cred = DefaultAzureCredential()
        _cred.get_token('https://cognitiveservices.azure.com/.default')
        _say(_OK, 'AAD token acquired (az login active)')
        _token_ok = True
    except Exception as e:
        _say(_WARN, f"couldn't get AAD token: {str(e)[:120]}  ->  run 'az login'")

# 5) Base model reachable (the real green light for inference) -------------
if _token_ok:
    dep = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
    try:
        _c = AzureOpenAI(
            azure_endpoint=endpoint,
            azure_ad_token_provider=lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
            api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        )
        _r = _c.chat.completions.create(model=dep, max_tokens=5, temperature=0,
            messages=[{'role': 'system', 'content': 'Reply with exactly: ready'}, {'role': 'user', 'content': 'ping'}])
        _say(_OK, f"base deployment '{dep}' responded: '{(_r.choices[0].message.content or '').strip()}'")
    except Exception as e:
        _say(_WARN, f"base deployment '{dep}' not reachable: {str(e)[:120]}")

# Verdict ------------------------------------------------------------------
print()
if 'data files' in _problems or 'sdk' in _problems:
    raise SystemExit('Preflight FAILED on a hard prerequisite above — fix it before continuing.')
if _token_ok and endpoint:
    print('GREEN: ready to run all RTOR labs (00 / 01 / 03 / 07).')
else:
    print('AMBER: local data path is ready. Set AZURE_OPENAI_ENDPOINT + run "az login" before Labs 01/03/07.')


---
## Step 1 — Load the rules and the labeled seed cases

In [ ]:
import json
from pathlib import Path

RULES = Path('data/rtor_rules.md').read_text(encoding='utf-8')
CASES = [json.loads(l) for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
pos = sum(1 for c in CASES if c['is_return_to_or'])
print(f'Rules KB: {len(RULES)} chars')
print(f'Labeled cases: {len(CASES)}  (RTOR=true: {pos}, RTOR=false: {len(CASES)-pos})')
print('\nExample case:', CASES[0]['case_id'])
print('  gold is_return_to_or:', CASES[0]['is_return_to_or'])
print('  gold evidence       :', CASES[0]['evidence'])


---
## Step 2 — The abstraction prompt (system rules + per-case template)

Exactly the shape of the production batch job: a rules-laden **system prompt** plus a **template** that injects each case's timeline and operative notes, asking for a strict two-key JSON answer.

In [ ]:
import json
from pathlib import Path

CASES = [json.loads(l) for l in Path('data/rtor_cases.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]

RULES_BLOCK = '''
### SPECIFIC ABSTRACTION RULES

Rule 1 - Conflict-resolution order (apply in this EXACT priority; the FIRST match decides):
  1. Planned / staged overrides everything. If the index OR current operative note documents that
     the second procedure was planned, staged, anticipated, or scheduled at the time of the index
     surgery, then is_return_to_or = false (even if it occurs within 30 days).
  2. Unplanned + related + within 30 days = RTOR. If the current surgery is unplanned and treats a
     complication of the index surgery (bleeding, hematoma, surgical-site infection, wound dehiscence,
     anastomotic leak, abscess, graft/flap failure) within 30 days, then is_return_to_or = true.
  3. Unrelated anatomy or new diagnosis = not RTOR (false), regardless of timing.
  4. Outside the 30-day window = not RTOR (false).

Rule 2 - Operating-room requirement. The return must be to an operating room. Bedside, ICU, IR,
  endoscopy-suite, or clinic procedures do NOT count: is_return_to_or = false.

Rule 3 - Evidence requirement. Quote the single most decisive sentence from the source documents
  verbatim, then state which rule it triggers.
'''

SYSTEM_PROMPT = (
    'You are a surgical-quality abstraction assistant for Acme Health. Determine whether the '
    'current operative episode is an unplanned Return to the Operating Room (RTOR) for the index '
    'surgery, applying the rules below.\n'
    + RULES_BLOCK +
    '\n### TASK EXECUTION\n'
    '- Read the provided text thoroughly.\n'
    '- Evaluate the context against the Specific Abstraction Rules, resolving any conflicting data '
    'using the exact order specified in Rule 1.\n'
    '- Output ONLY a valid JSON object with exactly two keys: "is_return_to_or" (boolean) and '
    '"evidence" (string citing the exact text used and how it applies to the rules).\n'
    '- Do not include conversational filler. Do not include markdown formatting like a json fence.'
)

TEMPLATE = '''Patient Timeline:
{patient_timeline_json}

Progress Note Details:
{progress_note_json}

Index Surgery Procedure Description:
{index_surgery_procedure_desc}

Index Surgery Operative Note:
{index_surgery_op_note}

Current Surgery Procedure Description:
{current_surgery_procedure_desc}

Current Surgery Operative Note:
{current_surgery_op_note}

Task: Determine if the current operating note/surgery represents a return to the operating room based strictly on the abstraction rules provided above. Output ONLY the raw JSON object.'''

def build_user_prompt(case):
    return TEMPLATE.format(
        patient_timeline_json        = json.dumps(case.get('patient_timeline', []), indent=2),
        progress_note_json           = json.dumps(case.get('progress_note', {}), indent=2),
        index_surgery_procedure_desc = case.get('index_surgery_procedure_desc', ''),
        index_surgery_op_note        = case.get('index_surgery_op_note', ''),
        current_surgery_procedure_desc = case.get('current_surgery_procedure_desc', ''),
        current_surgery_op_note      = case.get('current_surgery_op_note', ''),
    )

def safe_parse(val):
    '''Safely extract JSON from the LLM response, stripping stray markdown fences.'''
    try:
        clean = str(val).strip()
        if clean.startswith('```'):
            clean = clean.strip('`')
            if clean.startswith('json'):
                clean = clean[4:]
        return json.loads(clean.strip())
    except Exception as e:
        return {'is_return_to_or': None, 'evidence': f'Parse Error: {e} | Raw: {val}'}

print(f'Loaded {len(CASES)} labeled cases. Prompt + parser ready.')
print('--- USER PROMPT for', CASES[0]['case_id'], '(first 500 chars) ---')
print(build_user_prompt(CASES[0])[:500])


---
## Step 3 — (Optional, LIVE) augment with paraphrased variants

Use the deployed model to **reword the operative notes** while preserving every clinical fact, timing, and location — more training signal, same gold label. Safe to skip: if there are no Azure creds or the call fails, we keep just the seed cases.

In [ ]:
AUGMENT_VARIANTS_PER_CASE = 1
augmented = []
try:
    import os
    from dotenv import load_dotenv
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential
    load_dotenv()
    _cred = DefaultAzureCredential()
    _client = AzureOpenAI(
        azure_endpoint          = os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider = lambda: _cred.get_token('https://cognitiveservices.azure.com/.default').token,
        api_version             = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    )
    _dep = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')
    for case in CASES[:6]:
        for _ in range(AUGMENT_VARIANTS_PER_CASE):
            rw = _client.chat.completions.create(
                model=_dep, temperature=0.7, max_tokens=600,
                response_format={'type': 'json_object'},
                messages=[
                    {'role': 'system', 'content': 'You rewrite operative notes in different wording while preserving EVERY clinical fact, timing, and location. Reply with JSON containing only the keys index_surgery_op_note and current_surgery_op_note.'},
                    {'role': 'user', 'content': json.dumps({'index_surgery_op_note': case['index_surgery_op_note'], 'current_surgery_op_note': case['current_surgery_op_note']})},
                ],
            )
            nv = json.loads(rw.choices[0].message.content)
            variant = dict(case)
            variant['case_id'] = case['case_id'] + '-v'
            variant['index_surgery_op_note']   = nv.get('index_surgery_op_note', case['index_surgery_op_note'])
            variant['current_surgery_op_note'] = nv.get('current_surgery_op_note', case['current_surgery_op_note'])
            augmented.append(variant)
    print(f'Generated {len(augmented)} paraphrased variants (gold labels unchanged).')
except Exception as e:
    print(f'[augmentation skipped] {e}')

CASES_ALL = CASES + augmented
print('Total cases for the splits:', len(CASES_ALL))


---
## Step 4 — Emit the SFT splits and the eval set

Each training record is an OpenAI-style `messages` triple (system rules → case prompt → the gold JSON answer). The eval set keeps the raw case + gold label but **never leaks the answer into the prompt**, so Lab 07 scores honestly.

In [ ]:
import json, random
from pathlib import Path
random.seed(42)

def to_sft_record(case):
    answer = {'is_return_to_or': case['is_return_to_or'], 'evidence': case['evidence']}
    return {'messages': [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': build_user_prompt(case)},
        {'role': 'assistant', 'content': json.dumps(answer)},
    ]}

pool = list(CASES_ALL)
random.shuffle(pool)
val   = pool[:4]
train = pool[4:]

train_path = Path('data/rtor_training.jsonl')
val_path   = Path('data/rtor_validation.jsonl')
eval_path  = Path('data/rtor_eval.jsonl')

with open(train_path, 'w', encoding='utf-8-sig') as f:
    for c in train:
        f.write(json.dumps(to_sft_record(c)) + '\n')
with open(val_path, 'w', encoding='utf-8-sig') as f:
    for c in val:
        f.write(json.dumps(to_sft_record(c)) + '\n')
with open(eval_path, 'w', encoding='utf-8') as f:
    for c in CASES_ALL:
        rec = {k: c[k] for k in c if k != 'evidence'}
        rec['gold_is_return_to_or'] = c['is_return_to_or']
        rec['gold_evidence'] = c['evidence']
        f.write(json.dumps(rec) + '\n')

print(f'train: {len(train)}   val: {len(val)}   eval: {len(CASES_ALL)}')
print('Wrote:', train_path, '|', val_path, '|', eval_path)
print('\nSample SFT record (first 700 chars):')
print(json.dumps(to_sft_record(train[0]), indent=2)[:700], '...')


---
## Takeaways

- One rules file + a few labeled cases → a structured SFT dataset, no manual JSON wrangling.
- The **same** system prompt and template drive training, batch inference (Lab 03), and evaluation (Lab 07) — change the rules once, everything downstream follows.
- Next: **Lab 01** teaches the model to apply Rule 1's conflict ordering that base models routinely get wrong.